# Pipeline CRM — Reporte Oportunidades (Glue / Spark)

Produce **exclusivamente** `ReporteOportunidades` en Gold S3.
Lógica idéntica a `proc_reporte_oportunidades` del pipeline pandas.

In [ ]:
%idle_timeout 720
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5


In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from datetime import datetime
import boto3

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
spark.conf.set('spark.sql.session.timeZone', 'America/Mexico_City')
print('Spark session iniciada.')


In [ ]:
BUCKET      = 'aws-s3-reporte-atenea-usorg-nprd'
BASE        = f's3://{BUCKET}/data/'
GOLD_BASE   = f's3://{BUCKET}/data/Gold/Drive/'
FECHA_CORTE = '2026-07-01'

SOURCES = {
    'oportunidades':           'Silver/Atlas/ProcOportunidades/',
    'pedidos':                 'Silver/Atlas/ProcPedidos/',
    'casos':                   'Silver/Atlas/ProcCasosCrm/',
    'citas':                   'Silver/Atlas/ProcCitas/',
    'solicitudes_credito':     'Silver/Atlas/ProcSolicitudesCredito/',
    'historico_casos':         'Silver/Atlas/ProcCasosCrmHistorico/',
    'catalogo_usuarios':       'Silver/Atlas/ProcCatalogoUsuarios/',
    'historico_oportunidades': 'Silver/Atlas/ProcHistoricoOportunidades/',
    'historico_citas':         'Silver/Atlas/ProcHistoricoCitas/',
}

ID_AM_DUMMIES = [1067687, 1067696, 1067711, 1067720,
                 1067648, 1067654, 1067669, 1067678]

STATUS_PEDIDOS_ABIERTOS = [
    'revision de auto', 'auto en cita', 'negociacion de precio',
    'acuerdo de compraventa', 'formalizacion de compra', 'cita para entrega',
]

REPORTE_OPORTUNIDADES_COLS = [
    'opportunity_id','opportunity_name','opportunity_created_date',
    'opportunity_created_date_day','fecha_asignacion','opportunity_owner','owner_id',
    'opportunity_owner_equipo','nombre_comprador','id_am_comprador',
    'last_modified_date','opportunity_source','opportunity_source_aux','opportunity_stage',
    'flag_caso_tomado_perf_sc','asesor_caso_tomado_perf_sc','equipo_asesor_caso_tomado_perf_sc',
    'fecha_asignacion_perfilamiento_sc','fecha_caso_tomado_sc',
    'case_owner_name_perf_sc','case_owner_id_perf_sc','case_owner_equipo_perf_sc',
    'case_id_perfilamiento_sc','case_status_perfilamiento_sc',
    'perf_bc_score','perf_intencion_pago','perf_contactado','perf_interesado',
    'perf_comentarios','perf_canal_respuesta','perf_fecha_primer_contacto',
    'perf_fecha_ultimo_contacto','perf_contacto_otro_momento','perf_fecha_sig_contacto',
    'flag_perfilamento_credito','asesor_perfilamiento_credito',
    'asesor_perfilamiento_credito_id','equipo_asesor_perfilamiento_credito',
    'fecha_asignacion_perfilamiento_credito',
    'case_id_perfilamiento_credito','case_status_perfilamiento_credito',
    'cred_contactado','cred_perfilamiento_credit','cred_canal_respuesta_credito',
    'numero_pedidos','numero_pedidos_abiertos','fecha_primer_pedido','fecha_ultimo_pedido',
    'n_simulaciones_credito','n_simulaciones_con_folio',
    'n_simulaciones_preaceptadas','n_simulaciones_contingencia',
    'simulation_name_op','created_date_sim_op','tipo_credito',
    'flag_cita_agendada_oportunidad','flag_cita_sin_pedido_agendada',
    'flag_cita_sin_pedido_show','fecha_primera_cita_visita_comp',
    'fecha_ultima_cita_visita_comp','flag_cita_show_oportunidad',
    'stage_antes_de_cierre','usuario_que_cerro','equipo_que_cerro','fecha_cierre',
    'motivo_cierre_oportunidad','detalle_motivo_cierre_oportunidad',
    'primer_booker_comprador','equipo_primer_booker_comprador',
]


## 1. Lectura desde Silver DL

In [ ]:
def latest_partition(base_s3_path):
    s3 = boto3.client('s3')
    bucket, prefix = base_s3_path.replace('s3://', '').split('/', 1)
    prefix = prefix.rstrip('/') + '/'
    def max_partition(pfx, key):
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=pfx, Delimiter='/')
        parts = [(int(p['Prefix'].rstrip('/').split('=')[-1]), p['Prefix'])
                 for p in resp.get('CommonPrefixes', []) if '=' in p['Prefix']]
        if not parts: raise FileNotFoundError(f'Sin particiones ({key}=) en {pfx}')
        return max(parts)[1]
    yp = max_partition(prefix, 'year')
    mp = max_partition(yp,     'month')
    dp = max_partition(mp,     'day')
    full = f's3://{bucket}/{dp}'
    print(f'  -> {full}')
    return full


In [ ]:
_t0 = datetime.now()
_dfs = {}
for nombre, path in SOURCES.items():
    print(f'Leyendo {nombre}...')
    _dfs[nombre] = spark.read.parquet(latest_partition(BASE + path))
    print(f'  {_dfs[nombre].count():,} filas')

oportunidades           = _dfs['oportunidades']
pedidos                 = _dfs['pedidos']
casos                   = _dfs['casos']
citas                   = _dfs['citas']
solicitudes_credito     = _dfs['solicitudes_credito']
historico_casos         = _dfs['historico_casos']
catalogo_usuarios       = _dfs['catalogo_usuarios']
historico_oportunidades = _dfs['historico_oportunidades']
historico_citas         = _dfs['historico_citas']
print(f'Listo en {(datetime.now()-_t0).seconds}s')


In [ ]:
def filtrar_fecha(df, col, corte=FECHA_CORTE):
    if col not in df.columns:
        print(f'  [SKIP] {col!r} no encontrada')
        return df
    return df.filter(F.col(col) > F.lit(corte))

oportunidades           = filtrar_fecha(oportunidades,           'opportunity_created_date')
citas                   = filtrar_fecha(citas,                   'created_date')
historico_oportunidades = filtrar_fecha(historico_oportunidades, 'created_date')
historico_citas         = filtrar_fecha(historico_citas,         'created_date')
historico_casos         = filtrar_fecha(historico_casos,         'case_created_date')
casos                   = filtrar_fecha(casos,                   'case_created_date')


## 2. Helpers: catálogo SCD2

In [ ]:
# Catalogo: 1 fila por usuario (snapshot mas reciente)
if 'start_date' not in catalogo_usuarios.columns:
    catalogo_usuarios = (
        catalogo_usuarios
        .withColumn('start_date', F.lit('2000-01-01').cast('date'))
        .withColumn('end_date',   F.lit('2099-12-31').cast('date'))
    )

_w_cat = Window.partitionBy('id').orderBy(F.col('processing_date').desc())
_cat_dedup = (
    catalogo_usuarios
    .withColumn('_rn', F.row_number().over(_w_cat))
    .filter(F.col('_rn') == 1).drop('_rn')
)

def add_team_scd2(df, date_col, user_id_col, output_col):
    """Agrega equipo del catalogo vigente en date_col para user_id_col."""
    p = f'_scd_{output_col}'
    _cat = _cat_dedup.select(
        F.col('id').alias(f'{p}_id'),
        F.col('equipo').alias(output_col),
        F.col('start_date').alias(f'{p}_s'),
        F.col('end_date').alias(f'{p}_e'),
    )
    return (
        df.join(_cat, on=(
            (F.col(user_id_col).cast('string') == F.col(f'{p}_id').cast('string')) &
            (F.col(date_col).cast('date') >= F.col(f'{p}_s')) &
            (F.col(date_col).cast('date') <= F.col(f'{p}_e'))
        ), how='left')
        .drop(f'{p}_id', f'{p}_s', f'{p}_e')
    )
print('Catalogo y helper SCD2 listos.')


## 3. Enriquecimientos previos al join principal

In [ ]:
# Casos de perfilamiento Sales Center
_casos_sc = (
    casos
    .filter(F.lower(F.trim(F.col('case_subject'))) == 'perfilamiento contact center')
    .select(
        'opportunity_id',
        F.col('case_id').alias('case_id_perfilamiento_sc'),
        F.col('case_status').alias('case_status_perfilamiento_sc'),
        'case_owner_name_perf_sc',
        'case_owner_id_perf_sc',
    )
)

# Casos de perfilamiento Credito
_casos_cred = (
    casos
    .filter(F.lower(F.trim(F.col('case_subject'))) == 'perfilamiento credito')
    .select(
        'opportunity_id',
        F.col('case_id').alias('case_id_perfilamiento_credito'),
        F.col('case_status').alias('case_status_perfilamiento_credito'),
    )
)
print(f'casos SC: {_casos_sc.count():,} | credito: {_casos_cred.count():,}')


In [ ]:
# IDs de casos SC y credito para filtrar historico (broadcast-friendly)
_ids_sc   = [r.case_id_perfilamiento_sc   for r in _casos_sc.select('case_id_perfilamiento_sc').distinct().collect()]
_ids_cred = [r.case_id_perfilamiento_credito for r in _casos_cred.select('case_id_perfilamiento_credito').distinct().collect()]

# Quien tomo el caso SC: primer cambio status 'abierto' -> 'in progress'
_w_sc = Window.partitionBy('case_id').orderBy('created_date')
_asesor_sc = (
    historico_casos
    .filter(F.col('case_id').isin(_ids_sc))
    .filter(F.lower(F.trim(F.col('field')))     == 'status')
    .filter(F.lower(F.trim(F.col('old_value'))) == 'abierto')
    .filter(F.lower(F.trim(F.col('new_value'))) == 'in progress')
    .withColumn('_rn', F.row_number().over(_w_sc))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('case_id').alias('case_id_perfilamiento_sc'),
        F.col('created_date').alias('fecha_caso_tomado_sc'),
        F.upper(F.col('created_by')).alias('asesor_caso_tomado_perf_sc'),
        F.col('created_by_id').alias('_asesor_sc_id'),
    )
)

# Fecha de asignacion SC: primer cambio owner 'caso de servicio' -> otro
_w_asig = Window.partitionBy('case_id').orderBy('created_date')
_fecha_asig_sc = (
    historico_casos
    .filter(F.col('case_id').isin(_ids_sc))
    .filter(F.lower(F.trim(F.col('field')))     == 'owner')
    .filter(F.lower(F.trim(F.col('old_value'))) == 'caso de servicio')
    .filter(F.lower(F.trim(F.col('new_value'))) != 'caso de servicio')
    .withColumn('_rn', F.row_number().over(_w_asig))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('case_id').alias('case_id_perfilamiento_sc'),
        F.col('created_date').alias('fecha_asignacion_perfilamiento_sc'),
    )
)

# Quien tomo el caso credito: primer cambio status 'abierto' -> 'in progress'
_w_cred = Window.partitionBy('case_id').orderBy('created_date')
_asesor_cred = (
    historico_casos
    .filter(F.col('case_id').isin(_ids_cred))
    .filter(F.lower(F.trim(F.col('field')))     == 'status')
    .filter(F.lower(F.trim(F.col('old_value'))) == 'abierto')
    .filter(F.lower(F.trim(F.col('new_value'))) == 'in progress')
    .withColumn('_rn', F.row_number().over(_w_cred))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('case_id').alias('case_id_perfilamiento_credito'),
        F.col('created_date').alias('fecha_asignacion_perfilamiento_credito'),
        F.upper(F.col('created_by')).alias('asesor_perfilamiento_credito'),
        F.col('created_by_id').alias('asesor_perfilamiento_credito_id'),
    )
)
print('Historico casos procesado.')


In [ ]:
# 1 fila por opportunity: conteos y fechas de pedidos
_summary_ped = (
    pedidos
    .withColumn('_pedido_abierto',
        F.when(F.lower(F.trim(F.col('sf_order_status'))).isin(STATUS_PEDIDOS_ABIERTOS), 1).otherwise(0))
    .groupBy('opportunity_id')
    .agg(
        F.countDistinct('sf_order_id').alias('numero_pedidos'),
        F.sum('_pedido_abierto').alias('numero_pedidos_abiertos'),
        F.min('fecha_de_creacion').alias('fecha_primer_pedido'),
        F.max('fecha_de_creacion').alias('fecha_ultimo_pedido'),
    )
)
print(f'summary pedidos: {_summary_ped.count():,} opps con pedido')


In [ ]:
# 1 fila por opportunity: conteos de simulaciones
_summary_sol = (
    solicitudes_credito
    .withColumn('_folio_num', F.col('folio').cast('double'))
    .withColumn('_flag_folio',
        F.when(F.col('_folio_num').isNotNull() & (F.col('_folio_num') > 0), 1).otherwise(0))
    .withColumn('_flag_contingencia',
        F.when(F.lower(F.trim(F.col('tipo_conclusion_flujo_credito'))) == 'flujo contingente', 1).otherwise(0))
    .withColumn('_flag_aceptada',
        F.when(
            F.lower(F.coalesce(F.col('status_solicitud'), F.lit(''))).contains('acept') &
            (F.col('_flag_folio') == 1), 1).otherwise(0))
    .groupBy('opportunity_id')
    .agg(
        F.countDistinct('simulation_name').alias('n_simulaciones_credito'),
        F.sum('_flag_folio').alias('n_simulaciones_con_folio'),
        F.sum('_flag_aceptada').alias('n_simulaciones_preaceptadas'),
        F.sum('_flag_contingencia').alias('n_simulaciones_contingencia'),
    )
)
print(f'summary simulaciones: {_summary_sol.count():,} opps con simulacion')


In [ ]:
# Citas comprador vinculadas a oportunidad
_citas_comp = (
    citas
    .filter(F.col('opportunity_id').isNotNull())
    .filter(
        F.lower(F.coalesce(F.col('work_type_name'), F.lit(''))).isin(
            ['cita inicial visita comprador', ''])
        | F.col('work_type_name').isNull()
    )
    .withColumn('_cita_show',
        F.when(F.lower(F.trim(F.col('status'))).isin(['completa', 'en progreso']), 1).otherwise(0))
    .withColumn('_fecha_agend',
        F.date_format(F.col('sched_start_time').cast('date'), 'yyyy-MM-dd'))
)

_summary_citas = (
    _citas_comp
    .groupBy('opportunity_id')
    .agg(
        F.countDistinct('created_date').alias('_num_citas'),
        F.min('_fecha_agend').alias('fecha_primera_cita_visita_comp'),
        F.max('_fecha_agend').alias('fecha_ultima_cita_visita_comp'),
        F.sum('_cita_show').alias('_citas_show_sum'),
    )
    .withColumn('flag_cita_agendada_oportunidad',
        F.when(F.col('_num_citas') > 0, 1).otherwise(0))
    .withColumn('flag_cita_show_oportunidad',
        F.when(F.col('_citas_show_sum') > 0, 1).otherwise(0))
    .drop('_num_citas', '_citas_show_sum')
)

# Citas proactivas: sin opportunity_id y sin pedido asociado
_citas_proact = (
    citas
    .join(pedidos.select('commerce_order_id', 'sf_order_id').distinct(),
          on='commerce_order_id', how='left')
    .filter(F.col('opportunity_id').isNull())
    .filter(
        F.lower(F.coalesce(F.col('work_type_name'), F.lit(''))).isin(
            ['cita inicial visita comprador', ''])
        | F.col('work_type_name').isNull()
    )
    .filter(F.col('sf_order_id').isNull())
    .select('id_am', 'status').distinct()
)
_ids_proact      = {r.id_am for r in _citas_proact.select('id_am').collect() if r.id_am}
_ids_proact_show = {r.id_am for r in _citas_proact
    .filter(F.lower(F.trim(F.col('status'))).isin(['completa', 'en progreso']))
    .select('id_am').collect() if r.id_am}
print(f'summary citas: {_summary_citas.count():,} | proact: {len(_ids_proact)}')


In [ ]:
# Ultimo cierre 'cerrada (perdida)' por oportunidad + equipo de quien cerro (SCD2)
_w_cierre = Window.partitionBy('opportunity_id').orderBy(F.col('created_date').desc())
_hist_opp_mod = (
    historico_oportunidades
    .filter(F.lower(F.trim(F.col('new_value'))) == 'cerrada (perdida)')
    .withColumn('_rn', F.row_number().over(_w_cierre))
    .filter(F.col('_rn') == 1).drop('_rn')
    .withColumnRenamed('old_value',    'stage_antes_de_cierre')
    .withColumnRenamed('created_by',   'usuario_que_cerro')
    .withColumnRenamed('created_by_id','_id_usuario_que_cerro')
    .withColumnRenamed('created_date', 'fecha_cierre')
    .select('opportunity_id', 'stage_antes_de_cierre',
            'usuario_que_cerro', '_id_usuario_que_cerro', 'fecha_cierre')
)
_hist_opp_mod = add_team_scd2(
    _hist_opp_mod, 'fecha_cierre', '_id_usuario_que_cerro', 'equipo_que_cerro'
).drop('_id_usuario_que_cerro')
print(f'historico_opp cierres: {_hist_opp_mod.count():,}')


In [ ]:
# Primera cita comprador por oportunidad (por created_date asc)
_w_primera = Window.partitionBy('opportunity_id').orderBy(F.col('created_date').asc())
_primera_cita = (
    _citas_comp
    .withColumn('_rn', F.row_number().over(_w_primera))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select('opportunity_id', 'numero_cita')
)

# Quien agendo esa cita: primer status null -> 'scheduled' en historico_citas
_w_bk = Window.partitionBy('numero_cita').orderBy('created_date')
_booker = (
    historico_citas
    .filter(F.col('numero_cita').isin(
        [r.numero_cita for r in _primera_cita.select('numero_cita').collect()]))
    .filter(F.lower(F.trim(F.col('field')))     == 'status')
    .filter(F.col('old_value').isNull())
    .filter(F.lower(F.trim(F.col('new_value'))) == 'scheduled')
    .withColumn('_rn', F.row_number().over(_w_bk))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        'numero_cita',
        F.upper(F.col('created_by')).alias('primer_booker_comprador'),
        F.col('created_by_id').alias('_booker_id'),
        F.col('created_date').alias('_booker_fecha'),
    )
)

_primer_booker_df = (
    _primera_cita
    .join(_booker, on='numero_cita', how='left')
)
_primer_booker_df = add_team_scd2(
    _primer_booker_df, '_booker_fecha', '_booker_id', 'equipo_primer_booker_comprador'
).drop('numero_cita', '_booker_id', '_booker_fecha')
print(f'primer booker: {_primer_booker_df.count():,} opps')


## 4. Join principal + enrichments SCD2

In [ ]:
oportunidades_consumo = (
    oportunidades
    # equipo del owner de la oportunidad (SCD2)
    .transform(lambda df: add_team_scd2(
        df, 'opportunity_created_date', 'owner_id', 'opportunity_owner_equipo'))
    # caso SC
    .join(_casos_sc, on='opportunity_id', how='left')
    # fecha de asignacion SC
    .join(_fecha_asig_sc, on='case_id_perfilamiento_sc', how='left')
    # equipo del case_owner SC al momento de asignacion (SCD2)
    .transform(lambda df: add_team_scd2(
        df, 'fecha_asignacion_perfilamiento_sc', 'case_owner_id_perf_sc', 'case_owner_equipo_perf_sc'))
    # caso credito
    .join(_casos_cred, on='opportunity_id', how='left')
    # quien tomo el caso SC + su equipo (SCD2)
    .join(_asesor_sc, on='case_id_perfilamiento_sc', how='left')
    .transform(lambda df: add_team_scd2(
        df, 'fecha_caso_tomado_sc', '_asesor_sc_id', 'equipo_asesor_caso_tomado_perf_sc'))
    .drop('_asesor_sc_id')
    # quien tomo el caso credito + su equipo (SCD2)
    .join(_asesor_cred, on='case_id_perfilamiento_credito', how='left')
    .transform(lambda df: add_team_scd2(
        df, 'fecha_asignacion_perfilamiento_credito',
        'asesor_perfilamiento_credito_id', 'equipo_asesor_perfilamiento_credito'))
    .drop('asesor_perfilamiento_credito_id')
    # resumen pedidos (1:1)
    .join(_summary_ped, on='opportunity_id', how='left')
    # resumen simulaciones (1:1)
    .join(_summary_sol, on='opportunity_id', how='left')
    # resumen citas comprador (1:1)
    .join(_summary_citas, on='opportunity_id', how='left')
    # cierres y equipo_que_cerro
    .join(_hist_opp_mod, on='opportunity_id', how='left')
    # primer booker del comprador
    .join(_primer_booker_df, on='opportunity_id', how='left')
    # flags citas proactivas (broadcast desde sets Python)
    .withColumn('flag_cita_sin_pedido_agendada',
        F.when(F.col('id_am_comprador').cast('long').isin(list(_ids_proact)), 1).otherwise(0))
    .withColumn('flag_cita_sin_pedido_show',
        F.when(F.col('id_am_comprador').cast('long').isin(list(_ids_proact_show)), 1).otherwise(0))
    # flags de caso tomado y credito
    .withColumn('flag_caso_tomado_perf_sc',
        F.when(F.col('fecha_caso_tomado_sc').isNotNull(), 1).otherwise(0))
    .withColumn('flag_perfilamento_credito',
        F.when(F.col('fecha_asignacion_perfilamiento_credito').isNotNull(), 1).otherwise(0))
    # fecha de creacion solo-fecha (sin hora)
    .withColumn('opportunity_created_date_day',
        F.date_format(F.col('opportunity_created_date').cast('date'), 'yyyy-MM-dd'))
)
print(f'join principal: {oportunidades_consumo.count():,} filas')


## 5. Dedup + origen credito calculado + PUC

In [ ]:
# Dedup: 1 fila por opportunity_id. Preferencia a quien tenga fecha_asignacion_credito.
_w_dedup = Window.partitionBy('opportunity_id').orderBy(
    F.col('fecha_asignacion_perfilamiento_credito').desc_nulls_last())
oportunidades_consumo = (
    oportunidades_consumo
    .withColumn('_rn', F.row_number().over(_w_dedup))
    .filter(F.col('_rn') == 1).drop('_rn')
)
print(f'despues de dedup: {oportunidades_consumo.count():,} filas')


In [ ]:
# ------------------------------------------------------------------
# Origen credito calculado
# Para opps 'credito am' o 'apartado am': clasificar segun simulaciones
# ------------------------------------------------------------------
_FECHA_SAFETY = '2100-12-01'

# Solo opps objetivo
_ops_obj = oportunidades_consumo.filter(
    F.col('opportunity_source').isin(['credito am', 'apartado am'])
).select('opportunity_id', 'opportunity_source', 'fecha_caso_tomado_sc')

# Simulaciones de esas opps con flag_raro
_sims = (
    solicitudes_credito
    .join(_ops_obj.select('opportunity_id', 'fecha_caso_tomado_sc'),
          on='opportunity_id', how='inner')
    .withColumn('_flag_raro',
        F.when(
            F.col('created_date').cast('timestamp') >
            F.coalesce(F.col('fecha_caso_tomado_sc'), F.lit(_FECHA_SAFETY)).cast('timestamp'),
            1).otherwise(0))
)

# Contar sims y sims raras por opp
_w_counts = Window.partitionBy('opportunity_id')
_sims = (
    _sims
    .withColumn('n_sims',      F.countDistinct('simulation_name').over(_w_counts))
    .withColumn('n_flag_raro', F.sum('_flag_raro').over(_w_counts))
    # Quitar raras salvo que TODAS lo sean
    .filter((F.col('_flag_raro') != 1) | (F.col('n_sims') == F.col('n_flag_raro')))
    # Tier: 1=pre-aceptada, 2=rechazada-automatica, 3=contingencia
    .withColumn('_tier',
        F.when(F.lower(F.trim(F.col('status_solicitud'))) == 'pre aceptada', 1)
         .when(F.lower(F.trim(F.col('tipo_conclusion_flujo_credito'))) == 'flujo contingente', 3)
         .when(F.lower(F.trim(F.col('status_solicitud'))) == 'rechazada automatica', 2)
         .otherwise(F.lit(None).cast('int')))
)

# Mejor simulacion por opp (tier asc, fecha desc)
_w_mejor = Window.partitionBy('opportunity_id').orderBy(
    F.col('_tier').asc_nulls_last(), F.col('created_date').desc())
_mejor_sim = (
    _sims
    .withColumn('_rn', F.row_number().over(_w_mejor))
    .filter(F.col('_rn') == 1)
    .withColumn('_src_aux',
        F.when(F.lower(F.trim(F.col('tipo_conclusion_flujo_credito'))) == 'flujo contingente',
               F.lit('credito am contingencia'))
         .otherwise(F.lit('credito am api')))
    .withColumn('_suffix',
        F.when(
            (F.col('_src_aux') == 'credito am api') &
            F.lower(F.coalesce(F.col('status_solicitud'), F.lit(''))).contains('rechaz'),
            F.lit(' rechazado'))
         .when(F.col('_src_aux') == 'credito am api', F.lit(' aprobado'))
         .otherwise(F.lit('')))
    .withColumn('opportunity_source_calculado',
        F.concat(F.col('_src_aux'), F.col('_suffix')))
    .select(
        'opportunity_id',
        'opportunity_source_calculado',
        F.col('simulation_name').alias('simulation_name_op'),
        'tipo_credito',
        F.col('created_date').alias('created_date_sim_op'),
    )
)

oportunidades_consumo = (
    oportunidades_consumo
    .join(_mejor_sim, on='opportunity_id', how='left')
    .withColumn('opportunity_source_aux', F.col('opportunity_source_calculado'))
    .drop('opportunity_source_calculado')
)
print('Origen credito calculado OK.')


In [ ]:
# PUC: clasificacion de apartados sin simulacion
_score = F.col('perf_bc_score').cast('double')
_puc = (
    (F.col('opportunity_source') == 'apartado am') &
    (F.lower(F.trim(F.coalesce(F.col('perf_contactado'),  F.lit('')))) == 'si') &
    (F.lower(F.trim(F.coalesce(F.col('perf_interesado'),  F.lit('')))) == 'si') &
    (F.lower(F.trim(F.coalesce(F.col('perf_intencion_pago'), F.lit('')))).isin(
        ['crédito', 'credito'])) &
    F.col('perf_bc_score').isNotNull() &
    F.col('opportunity_source_aux').isNull()
)
oportunidades_consumo = oportunidades_consumo.withColumn(
    'opportunity_source_aux',
    F.when(_puc & (_score <= 569),              F.lit('apartado puc no apto'))
     .when(_puc & _score.between(570, 649),     F.lit('apartado puc viable kuna'))
     .when(_puc & _score.between(650, 699),     F.lit('apartado puc viable bbva'))
     .when(_puc & (_score >= 700),              F.lit('apartado puc directo eam'))
     .otherwise(F.col('opportunity_source_aux'))
)

# Filtros finales: fecha de corte y excluir dummies
oportunidades_consumo = (
    oportunidades_consumo
    .filter(F.col('opportunity_created_date_day') >= FECHA_CORTE)
    .filter(~F.col('id_am_comprador').cast('long').isin(ID_AM_DUMMIES))
)

# Seleccionar solo columnas de salida que existan en el df
_cols_out = [c for c in REPORTE_OPORTUNIDADES_COLS if c in oportunidades_consumo.columns]
_cols_faltantes = [c for c in REPORTE_OPORTUNIDADES_COLS if c not in oportunidades_consumo.columns]
if _cols_faltantes:
    print(f'  [INFO] columnas no disponibles en Silver DL ({len(_cols_faltantes)}): {_cols_faltantes}')
oportunidades_consumo = oportunidades_consumo.select(_cols_out)
print(f'oportunidades_consumo FINAL: {oportunidades_consumo.count():,} filas, {len(_cols_out)} cols')


## 6. Escritura en Gold S3

In [ ]:
path = GOLD_BASE + 'ReporteOportunidades/'
oportunidades_consumo.write.mode('overwrite').parquet(path)
print(f'Escrito ReporteOportunidades -> {path}')
print(f'Pipeline completado: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
